In [ ]:
## imports module
import anndata as ad
import scanpy as sc
import gc, os
import sys
from rich import print
import numpy as np
import psutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sea
import harmonypy as hm
import os
import psutil, gc

from rich import print

In [ ]:
## general setting
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80)
pd.set_option('display.max_columns', None)

# check how many memory used
print(psutil.virtual_memory())
# Force garbage collection
gc.collect()  
sc.settings.set_figure_params(dpi=200, facecolor="white")
PATH = "/home/kibr/Work/Vascular_atlas"

## set path and parameters
os.chdir(PATH)
sc.set_figure_params(figsize=(6, 6), frameon=False)

In [ ]:
## --- Load data --- ##
## Setting one: all endothelial cells
adata = sc.read_h5ad(PATH + "/Results/Revision_2/clean_object_revision_03032026.h5ad")#vnoised',color="cell.class_after_decontX", legend_loc="on data")
adata = adata[adata.obs['Cell_class'].isin(['Endothelial'])].copy()
## Show the data
print(adata)
print(adata.X[:10,:10])

In [ ]:
## save backup
adata.raw = adata.copy()

## filter the features again
sc.pp.filter_genes(adata, min_cells=20) 

print(adata)
print(adata.X[:10,:10])

### Performing another round of integration using Harmony

In [ ]:
## normalization
sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

print(adata.X[:8,:8])

In [ ]:
## to reach the best results, using the below setting
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    subset=False,
    layer="log1p",
    flavor="seurat",
    batch_key="individualID",
)

## only using the highly variable genes for integration task
adata = adata[:,adata.var["highly_variable"]].copy()
print(adata)

In [ ]:
### pca and integration
print("Running PCA")
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")
print(psutil.virtual_memory())

In [ ]:
## Harmony integration 
print(adata)
print("Applying Harmony integration")
sc.external.pp.harmony_integrate(adata, key='individualID', max_iter_harmony=50,basis='X_pca')
rep = "X_pca_harmony"

print(psutil.virtual_memory())

In [ ]:
print("Computing neighbors")
rep = "X_pca_harmony"
sc.pp.neighbors(adata, use_rep=rep,metric="cosine")

print("Computing leiden")
sc.tl.leiden(adata,resolution=0.5,key_added='leiden_harmony')

# Check integration figures for remaining batch(individual effect)
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures"
os.makedirs(sc.settings.figdir, exist_ok=True)

sc.tl.umap(
    adata,
    # reuse existing neighbors graph
    random_state=42,
    key_added="umap_harmony"
)

sc.pl.embedding(
    adata,
    basis="umap_harmony", 
    # color=["leiden_harmony", "individualID",'brain_region'],
    color=["leiden_harmony", "individualID",'brain_region',"Cell_type"],
    frameon=False,
    ncols=4,
    size=20,
    legend_loc="on data",
    # save=f"_Endo_umap_harmony.svg"
)

In [ ]:
### Given some known cell type markers, annotate the clusters
adata = adata.raw.to_adata()
adata.raw = adata.copy()
print(adata.X[:10,:10])

sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10)

markers = {'Large_Artery':'DKK2','Arterial':"VEGFC",'Capillary':"SLC7A5",'Venous':"RAMP3",'Fenestrated':'PLVAP','EndoMT':'MKI67'}
sc.pl.dotplot(adata, markers, groupby='leiden_harmony')

In [ ]:
## Assign cell type annotation based on the dotplot and known markers
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures"
# cell types
cluster2celltype = {
    "0": "Capillary",
    "1": "Capillary",
    "2": "Venous",
    "3": "Arterial",
    "4": "Capillary",
    "5": "Large_Artery",
    "6": "Fenestrated_Capillary",
    "7": "EndoMT",}

adata.obs["Cell_type"] = adata.obs["leiden_harmony"].map(cluster2celltype)

sc.pl.embedding(
    adata,
    basis='umap_harmony',   # <- THIS picks .obsm['X_' + key]
    color=['leiden_harmony','individualID','Cell_type'],
    frameon=False,
    ncols=4,
    size=20,
    legend_loc="on data",
    use_raw = False,
    save=f"_Endo_CT.svg"
    )


In [ ]:
markers = {'Large_Artery':'DKK2','Arterial':"VEGFC",'Capillary':"SLC7A5",'Venous':"RAMP3",'Fenestrated':'PLVAP','EndoMT':'MKI67'}
# sc.pl.dotplot(adata, markers, groupby='Cell_type',use_raw=False)

sc.pl.dotplot(adata, markers, groupby='Cell_type',
              categories_order=['Large_Artery', 'Arterial', 'Capillary', 'Venous', 'Fenestrated_Capillary', 'EndoMT'],
              use_raw=False)

In [ ]:
### ---------------------------------------------------------------------
### Saving processed data
### ---------------------------------------------------------------------
adata = adata.raw.to_adata()

print(adata)
print(adata.X[:10,:10])
## Saving h5ad
results_file = PATH+"/Results/Revision_2/Endo_temp_processed.h5ad"
adata.write(results_file)
print("Done. Saved:", results_file)

## Builting region-level representation (Donor-balanced centroids in PC spave)

In [ ]:
### Reloading the data 
adata = sc.read_h5ad(PATH+"/Results/Revision_2/Endo_temp_processed.h5ad")
print(adata)
print(adata.X[:10,:10])

sc.pl.embedding(
    adata,
    basis='umap_harmony',   # <- THIS picks .obsm['X_' + key]
    color=['leiden_harmony','individualID','Cell_type'],
    frameon=False,
    ncols=4,
    size=20,
    legend_loc="on data",
    use_raw = False,
    save=f"_Endo_CT.svg"
    )

In [ ]:
## check the cell type distribution across all brain regions
meta = adata.obs
pd.crosstab(meta['Cell_type'],meta['brain_region'])

In [ ]:
meta_sub = meta[meta['Cell_type'] == "Fenestrated_Capillary"]
# meta_sub
pd.crosstab(meta_sub['brain_region'],meta_sub['individualID'])

In [ ]:
# Choose how many PCs to represent each cell
rep = "X_pca_harmony"
n_pcs_use = 30
Xpc = adata.obsm[rep][:, :n_pcs_use]

# centroid for each (region, donor) group
df_index = adata.obs[["brain_region", "individualID"]].copy()
df_index["cell_idx"] = np.arange(adata.n_obs)

# Compute group centroids efficiently
# We'll build a table of centroids: rows = (region, donor), cols = PC1..PCn
pcs = pd.DataFrame(Xpc, index=adata.obs_names, columns=[f"PC{i+1}" for i in range(n_pcs_use)])
pcs["brain_region"] = adata.obs["brain_region"].values
pcs["individualID"]  = adata.obs["individualID"].values

rg_dn_centroids = pcs.groupby(["brain_region", "individualID"]).mean()
# print(rg_dn_centroids.head())

## donor-balanced region centroid
rg_centroids = rg_dn_centroids.groupby(level=0).mean()

# Optional: track how many donors contribute per region (useful QC)
donors_per_region = rg_dn_centroids.reset_index().groupby("brain_region")["individualID"].nunique()

## Create region -level AnnData and run leiden on region graph
adata_rg = sc.AnnData(X=rg_centroids.values)
adata_rg.obs_names = rg_centroids.index.astype(str)
adata_rg.var_names = [f"PC{i+1}" for i in range(n_pcs_use)]

adata_rg.obs["n_donors"] = donors_per_region.reindex(adata_rg.obs_names).fillna(0).astype(int).values

print(adata_rg)

## Adding meta data

In [ ]:
## Get the region ID and region name from the obs_names
adata_rg.obs["region_ID"] = (
    adata_rg.obs_names
        .astype(str)
        .str.split("_", n=1)
        .str[0]
)
adata_rg.obs["region_name"] = (
    adata_rg.obs_names
        .astype(str)
        .str.split("_", n=1)
        .str[1]
)

## read the region metadata
df = pd.read_csv(PATH + '/Data/region.csv')
# df.head()

# check for mismatches
df["regionID"] = df["regionID"].astype(str).str.strip()
adata_rg.obs["region_ID"] = adata_rg.obs["region_ID"].astype(str).str.strip()

missing = set(adata_rg.obs["region_ID"]) - set(df["regionID"])
print("Region IDs missing from metadata:", missing)

## adding annotation
adata_rg.obs["Region_layer"] = (
    adata_rg.obs["region_ID"]
        .map(df.set_index("regionID")["Region_layer_4"])
)

adata_rg.obs["Region_full_name"] = (
    adata_rg.obs["region_ID"]
        .map(df.set_index("regionID")["Region_layer_1"])
)

## Capitalize the region names for better visualization
adata_rg.obs["Region_full_name"] = adata_rg.obs["Region_full_name"].str.title()
## string combine region_ID and region_name for better visualization
adata_rg.obs["Region_combined"] = adata_rg.obs["region_ID"] + "_" + adata_rg.obs["Region_full_name"]
adata_rg.obs["Region_combined_2"] = adata_rg.obs["region_ID"] + "_" + adata_rg.obs["region_name"]

adata_rg.obs.set_index('Region_combined', inplace=True)

adata_rg.obs["Region_combined"] = adata_rg.obs.index.copy()

adata_rg.obs.head()

### Clustering and plotting

In [ ]:
# Build kNN graph across ~40 regions
# With only 40 nodes, keep neighbors modest (e.g., 5–15)
sc.pp.neighbors(adata_rg, n_neighbors=5, n_pcs=n_pcs_use, use_rep="X")

# Leiden clustering of regions (sweep resolution later)
sc.tl.leiden(adata_rg, resolution=1.5, key_added="leiden")

# 2D embedding for visualization 
sc.tl.umap(adata_rg, min_dist=0.3, random_state=42)

sc.tl.dendrogram(adata_rg, groupby="leiden")

## working on the dot size based on nuclei counts
### Clean the region names for better visualization
adata.obs["regionID"] = adata.obs["regionID"].astype(str).str.strip()
adata.obs["region_name"] = (adata.obs["regionID"].map(df.set_index("regionID")["Region_layer_1"]))

## Capitalize the region names for better visualization
adata.obs["region_name"] = adata.obs["region_name"].str.title()
## string combine regionID and region_name for better visualization
adata.obs["Region_combined"] = adata.obs["regionID"].astype(str) + "_" + adata.obs["region_name"]
## Get the nuclei counts for each region
nuclei_counts = adata.obs['Region_combined'].value_counts()

## ordering counts based on adata_rg.obs['Region_combined']
nuclei_counts_dict = nuclei_counts.to_dict()

# Size parameters might require scaling; you can adjust the scaling factor as needed
size_factor = 2  # Adjust this factor to change the size scale
dot_sizes = np.array([nuclei_counts_dict[region] * size_factor for region in adata_rg.obs.index])
print(nuclei_counts.head())

In [ ]:
# ----------------------------
# Plotting (region-level)
# ----------------------------
sc.set_figure_params(figsize=(8, 8), frameon=False)

sc.pl.umap(
    adata_rg,
    color="leiden",
    size=dot_sizes,
    legend_loc="on data",
    show=False
)

# add region labels
ax = plt.gca()
X = adata_rg.obsm["X_umap"]

for i, region in enumerate(adata_rg.obs_names):
    label = region.split("_", 1)[-1]
    ax.text(
        X[i, 0],
        X[i, 1],
        label,
        fontsize=8,
        ha="center",
        va="center"
    )

plt.show()

## UMAP of region_layer
sc.pl.umap(
    adata_rg,
    color="Region_layer",
    size=dot_sizes,
    legend_loc="right margin",
    show=False
)

# add region labels
ax = plt.gca()
X = adata_rg.obsm["X_umap"]

for i, region in enumerate(adata_rg.obs_names):
    label = region.split("_", 1)[-1]
    ax.text(
        X[i, 0],
        X[i, 1],
        label,
        fontsize=8,
        ha="center",
        va="center"
    )

plt.show()

# sc.pl.umap(adata_rg, color=["region_name"],legend_loc = "on data")
sc.pl.dendrogram(adata_rg, groupby="leiden",)

In [ ]:
# ----------------------------
# 1) Define clusters (leiden -> region_cluster)
# ----------------------------
cluster_map = {
    "4": "Cluster_4",
    "0": "Cluster_2",
    "1": "Cluster_3",
    "2": "Cluster_3",
    "3": "Cluster_1",
    "5": "Cluster_5",
}

adata_rg.obs["region_cluster"] = adata_rg.obs["leiden"].astype(str).map(cluster_map).fillna("Unassigned").astype("category")

# ----------------------------
# 2) Define Region_layer colors (your palette)
# ----------------------------
region_color_map = {
    "Cerebral Cortex": "#009E73",
    "Brain Stem and Spinal Cord": "#D55E00",
    "Cerebellum": "#046299",
    "Subcortical Region": "#03B8D8",
    "Cerebral Cortex Watershed": "#E69F00",
    "White Matter": "#CC79A7",
    "Olfactory Bulb": "#999999",
    "Choroid Plexus": "#9C0AE0",
    "Leptomeninges": "#915330",
    "Major Vessel": "#FF0000",
}

# Ensure categorical for stable color ordering
adata_rg.obs["Region_layer"] = adata_rg.obs["Region_layer"].astype("category")

# Scanpy expects a list of colors aligned to category order
adata_rg.uns["Region_layer_colors"] = [
    region_color_map.get(cat, "#BDBDBD")  
    for cat in adata_rg.obs["Region_layer"].cat.categories
]

# ----------------------------
# 3) Plot UMAP colored by Region_layer
# ----------------------------
sc.set_figure_params(figsize=(9, 9), frameon=False)

sc.pl.umap(
    adata_rg,
    color="Region_layer",
    size=dot_sizes,
    legend_loc="right margin",
    show=False
)

ax = plt.gca()
X = adata_rg.obsm["X_umap"]

# ----------------------------
# 4) Add region labels (after the first "_")
# ----------------------------
for (x, y), region in zip(X, adata_rg.obs_names):
    label = region.split("_", 1)[-1]
    ax.text(
        x, y, label,
        fontsize=10,
        ha="center",
        va="center",
        clip_on=True
    )

# ----------------------------
# 5) Add KDE contours per region_cluster + label each contour
# ----------------------------
clusters = adata_rg.obs["region_cluster"].astype(str)

# Tuning knobs for "closer" contours:
bw_adjust = 1.2   # smaller -> tighter / more detailed
thresh = 0.15     # larger -> tighter (keeps higher-density region)
alpha = 0.1  # smaller -> more transparent contours (can help with overlap)

for cl in sorted(clusters.unique()):
    if cl == "Unassigned":
        continue

    idx = (clusters == cl).values
    pts = X[idx, :]

    # KDE unstable for very small clusters
    if pts.shape[0] < 3:
        continue

    sea.kdeplot(
        x=pts[:, 0],
        y=pts[:, 1],
        fill=True,          # << shaded region
        levels=4,           # single filled region
        bw_adjust=bw_adjust,
        thresh=thresh,
        alpha=alpha,
        linewidth=0,
        ax=ax
    )

    # Optional: cluster label at center
    cx, cy = np.median(pts[:, 0]), np.median(pts[:, 1])
    ax.text(
        cx, cy, cl,
        fontsize=11,
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.2", alpha=0.6, linewidth=0),
        clip_on=True
    )

# # Optional cleanup
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_aspect("equal")
plt.savefig("./Results/Revision_2/Figures/umap_region_clusters_full_name_Endo.svg",format="svg",bbox_inches="tight",dpi=1200)
plt.show()

In [ ]:
## Option 2
# ----------------------------
# 1) Define clusters (leiden -> region_cluster)
# ----------------------------
cluster_map = {
    "4": "Cluster_4",
    "0": "Cluster_2",
    "1": "Cluster_3",
    "2": "Cluster_3",
    "3": "Cluster_1",
    "5": "Cluster_5",
}

adata_rg.obs["region_cluster"] = adata_rg.obs["leiden"].astype(str).map(cluster_map).fillna("Unassigned").astype("category")

# ----------------------------
# 2) Define Region_layer colors (your palette)
# ----------------------------
region_color_map = {
    "Cerebral Cortex": "#009E73",
    "Brain Stem and Spinal Cord": "#D55E00",
    "Cerebellum": "#046299",
    "Subcortical Region": "#03B8D8",
    "Cerebral Cortex Watershed": "#E69F00",
    "White Matter": "#CC79A7",
    "Olfactory Bulb": "#999999",
    "Choroid Plexus": "#9C0AE0",
    "Leptomeninges": "#915330",
    "Major Vessel": "#FF0000",
}


# Ensure categorical for stable color ordering
adata_rg.obs["Region_layer"] = adata_rg.obs["Region_layer"].astype("category")

# Scanpy expects a list of colors aligned to category order
adata_rg.uns["Region_layer_colors"] = [
    region_color_map.get(cat, "#BDBDBD")  
    for cat in adata_rg.obs["Region_layer"].cat.categories
]

# ----------------------------
# 3) Plot UMAP colored by Region_layer
# ----------------------------
sc.set_figure_params(figsize=(9, 9), frameon=False)

sc.pl.umap(
    adata_rg,
    color="Region_layer",
    size=dot_sizes,
    legend_loc="right margin",
    show=False
)

ax = plt.gca()
X = adata_rg.obsm["X_umap"]

# ----------------------------
# 4) Add region labels (after the first "_")
# ----------------------------
for (x, y), region in zip(X, adata_rg.obs_names):
    label = adata_rg.obs[adata_rg.obs_names == region]['region_name'].astype(str).values[0]
    ax.text(
        x, y, label,
        fontsize=10,
        ha="center",
        va="center",
        clip_on=True
    )
# ----------------------------
# 5) Add KDE contours per region_cluster + label each contour
# ----------------------------
clusters = adata_rg.obs["region_cluster"].astype(str)

# Tuning knobs for "closer" contours:
bw_adjust = 1.2   # smaller -> tighter / more detailed
thresh = 0.15     # larger -> tighter (keeps higher-density region)
alpha = 0.1  # smaller -> more transparent contours (can help with overlap)

for cl in sorted(clusters.unique()):
    if cl == "Unassigned":
        continue

    idx = (clusters == cl).values
    pts = X[idx, :]

    # KDE unstable for very small clusters
    if pts.shape[0] < 3:
        continue

    sea.kdeplot(
        x=pts[:, 0],
        y=pts[:, 1],
        fill=True,          # << shaded region
        levels=4,           # single filled region
        bw_adjust=bw_adjust,
        thresh=thresh,
        alpha=alpha,
        linewidth=0,
        ax=ax
    )

    # Optional: cluster label at center
    cx, cy = np.median(pts[:, 0]), np.median(pts[:, 1])
    ax.text(
        cx, cy, cl,
        fontsize=11,
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.2", alpha=0.6, linewidth=0),
        clip_on=True
    )

# # Optional cleanup
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_aspect("equal")

plt.savefig("./Results/Revision_2/Figures/umap_region_clusters_abb_Endo.svg",format="svg",bbox_inches="tight",dpi=600)
plt.show()

### Draw Dot plot for region clusters

In [ ]:
## Dot plot to show the belongness using the same color scheme for region layer UMAP, but x axis is the leiden cluster and y axis is the region name
df = adata_rg.obs.copy()
df["region"] = adata_rg.obs_names

plt.figure(figsize=(6, len(df) * 0.25))
sea.scatterplot(
    data=df,
    x="leiden",
    y="region",
    hue="Region_layer",
    # style="Region_layer",
    s=200
)
plt.xlabel("Leiden cluster")
plt.ylabel("Brain region")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("./Results/Revision_2/Figures/Region_clustering_Endo_belongness_dot.svg",format="svg",bbox_inches="tight",dpi=600)
plt.show()

In [ ]:
### Mapping the cluster labels back to single-cell data
# Create a mapping from region to leiden cluster
adata_rg.obs['brain_region'] = adata_rg.obs['Region_combined_2']
region_to_leiden = adata_rg.obs.set_index("brain_region")["leiden"].to_dict()
region_to_cluster = adata_rg.obs.set_index("brain_region")["region_cluster"].to_dict()
adata.obs["region_leiden"] = adata.obs["brain_region"].map(region_to_leiden)
adata.obs["region_cluster"] = adata.obs["brain_region"].map(region_to_cluster)

# region_cluster

In [ ]:
adata.obs["region_cluster"].value_counts()

In [ ]:
sc.pl.embedding(adata, basis="umap_harmony", color=['leiden_harmony',"brain_region", "region_cluster","Cell_type"],frameon=False, size=20,legend_loc="on data",)

In [ ]:
pd.crosstab(adata.obs["region_cluster"], adata.obs["individualID"])

In [ ]:
print(adata)
print(adata.X[:10,:10])

In [ ]:
### ---------------------------------------------------------------------
### Saving processed data
### ---------------------------------------------------------------------
# adata = adata.raw.to_adata()
print(adata)
## Saving h5ad
results_file = PATH+"/Results/Revision_2/Endo_temp_processed.h5ad"
adata.write(results_file)
print("Done. Saved:", results_file)

In [ ]:
results_file = PATH+"/Results/Revision_2/Endo_temp_processed.h5ad"
adata = sc.read_h5ad(results_file)
print(adata)

In [ ]:
cols = ['region_name', 'region_layer', 'region_cluster']

df_unique = (
    adata.obs[cols]
    .drop_duplicates()
)
display(df_unique)
df_unique.to_csv("./Results/Revision_2/HUMAN_unique_region_metadata.csv", index=False)

## Adding module score of BBB functions

In [ ]:
### Reloading the data
print(adata)
print(adata.X[:10,:10])

### normalization and log1p for plotting
sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)

In [ ]:
### Read in the gene list from GO_term.gaf file
go_file = PATH + "/Data/GO_terms/transport across blood-brain barrier.gaf"
go_df = pd.read_csv(go_file, sep="\t", comment="!", header=None)
go_df = go_df.iloc[:, :3]  # Keep only the first 3 columns
go_df.columns = [
    "DB",
    "DB_Object_ID",
    "Gene_Symbol"]  
go_df.head()

gene_list = go_df["Gene_Symbol"].unique().tolist()
print(f"Total unique genes in GO term: {len(gene_list)}") 

In [ ]:
### Adding module score to the adata object
# Check how many genes from the list are in the dataset
genes_in_data = [gene for gene in gene_list if gene in adata.var_names]
print(f"Genes from GO term present in data: {len(genes_in_data)}")  
# Calculate module score
sc.tl.score_genes(adata, gene_list=genes_in_data, score_name="BBB_Transport_Score", use_raw=False)

In [ ]:
## Plotting the module score on UMAP
## setting the color map to a more visually appealing one (e.g., "viridis" or "plasma")
## change the size of the figeure and the dot size for better visualization

## Assign cell type annotation based on the dotplot and known markers
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures"
sc.settings.set_figure_params(figsize=(8, 6), frameon=False)

sc.pl.embedding(
    adata,
    basis="umap_harmony",
    color="BBB_Transport_Score",
    frameon=False,
    cmap="magma",
    size=50,
    legend_loc="right margin",
    show=True,
    save=f"_BBB_Transport_Score.svg"
)


In [ ]:
adata.obs['Cell_type'].value_counts()

In [ ]:
## plottint the celtype with specific color map
sc.settings.set_figure_params(figsize=(8, 6), frameon=False)

celltype_colors = {
    'Large_Artery': '#B2182B',
    'Arterial': '#F4A582',
    'Capillary': '#66C2A5',
    'Venous': '#4393C3',
    'Fenestrated_Capillary': '#FDAE61',
    'EndoMT': '#BC80BD'
}

# enforce category order
adata.obs['Cell_type'] = adata.obs['Cell_type'].cat.reorder_categories(
    celltype_colors.keys(), ordered=True
)

# assign colors
adata.uns['Cell_type_colors'] = list(celltype_colors.values())


sc.pl.embedding(
    adata,
    basis='umap_harmony',   # <- THIS picks .obsm['X_' + key]
    color=['Cell_type'],
    frameon=False,
    ncols=4,
    size=20,
    # legend_loc="on data",
    use_raw = False,
    save=f"_Endo_CT_only.svg"
    )

In [ ]:
### Draw a bubble plot with the same color scheme for cell types, but size based on the BBB_Transport_Score
markers = {'Large_Artery':'DKK2','Arterial':"VEGFC",'Capillary':"SLC7A5",'Venous':"RAMP3",'Fenestrated':'PLVAP','EndoMT':'MKI67'}
# sc.pl.dotplot(adata, markers, groupby='Cell_type',use_raw=False)

sc.pl.dotplot(adata, markers, groupby='Cell_type',
              categories_order=['Large_Artery', 'Arterial', 'Capillary', 'Venous', 'Fenestrated_Capillary', 'EndoMT'],
              use_raw=False,
              save=f"_Endo_CT_dotplot.svg"
              )

In [ ]:
### Draw a violin plot showing the distribution of BBB_Transport_Score across different cell types
plt.figure(figsize=(8, 6))
sea.violinplot(
    data=adata.obs,
    x="Cell_type",
    y="BBB_Transport_Score",
    palette=celltype_colors,
    order=celltype_colors.keys()
)
plt.xlabel("Cell Type")
plt.ylabel("BBB Transport Score")
plt.title("Distribution of BBB Transport Score across Cell Types")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("./Results/Revision_2/Figures/BBB_Transport_Score_boxplot_Endo.svg",format="svg",bbox_inches="tight",dpi=600)
plt.show()

In [ ]:
### Check the region clusters on the UMAP of region centroids
results_file = PATH+"/Results/Revision_2/Endo_temp_processed.h5ad"
adata = sc.read_h5ad(results_file)
print(adata)

df = pd.crosstab(adata.obs['region_abb'], adata.obs['region_cluster'])
df[df>0] = 1
df